In [2]:
import pandas as pd
import numpy as np
print(pd.__version__, np.__version__)

3.0.6 2.4.6


In [11]:
df = pd.read_csv("../data/auth_logs.csv", parse_dates=["time"])
df.head()

,time,ip,user,event,status
0,2026-10-01 08:00:01,10.0.0.13,erin,login,success
1,2026-10-01 08:00:13,10.0.0.16,dave,login,success
2,2026-10-01 08:00:18,10.0.0.25,alice,login,success
3,2026-10-01 08:00:27,10.0.0.21,alice,login,success
4,2026-10-01 08:00:32,10.0.0.17,dave,login,success


In [12]:
df.shape, df.dtypes

((3200, 5),
 time      datetime64[us]
 ip                   str
 user                 str
 event                str
 status               str
 dtype: object)

In [13]:
df["status"].value_counts()

status
success    2865
failed      335
Name: count, dtype: int64

In [14]:
failed = df[df["status"] == "failed"]
failed["ip"].value_counts().head()

ip
203.0.113.50    200
10.0.0.25        10
10.0.0.24        10
10.0.0.27         9
10.0.0.13         8
Name: count, dtype: int64

In [15]:
atk = df[df["ip"] == "203.0.113.50"]
print(atk["time"].min(), atk["time"].max())
print(atk["user"].nunique(), len(atk))

2026-10-01 11:00:03 2026-10-01 11:04:58
5 200


In [16]:
failed.groupby("ip").size().sort_values(ascending=False).head(10)
failed.groupby("user").size()

user
alice    73
bob      62
carol    64
dave     64
erin     72
dtype: int64

In [17]:
w = (failed
     .groupby([pd.Grouper(key="time", freq="5min"), "ip"])
     .agg(n_failed=("status", "size"), n_users=("user", "nunique"))
     .reset_index())
w.sort_values("n_failed", ascending=False).head()

,time,ip,n_failed,n_users
61,2026-10-01 11:00:00,203.0.113.50,200,5
87,2026-10-01 12:05:00,10.0.0.26,2,1
7,2026-10-01 08:15:00,10.0.0.29,2,2
106,2026-10-01 13:00:00,10.0.0.11,2,1
123,2026-10-01 13:45:00,10.0.0.20,2,2


In [18]:
w["is_attack"] = (w["ip"] == "203.0.113.50").astype(int)
w["rule_flag"] = (w["n_failed"] > 20).astype(int)
pd.crosstab(w["is_attack"], w["rule_flag"])

rule_flag,0,1
is_attack,,
0,128,0
1,0,1
